In [ ]:
!pip install Transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
from google.colab import drive

# Connect Google Drive to Google Colab
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/Dataset_ap/training_checkpoints"
MODEL_PATH = "/content/drive/MyDrive/Dataset_ap/saved_summary_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Checkpoint directory:", OUTPUT_DIR)
print("Directory exists:", os.path.exists(OUTPUT_DIR))

Checkpoint directory: /content/drive/MyDrive/Dataset_ap/training_checkpoints
Directory exists: True


In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/Dataset_ap"

train_path = os.path.join(DATASET_PATH, "samsum-train.csv")
validation_path = os.path.join(DATASET_PATH, "samsum-validation.csv")
test_path = os.path.join(DATASET_PATH, "samsum-test.csv")

for path in [train_path, validation_path, test_path]:
    print(f"{os.path.basename(path)}: {os.path.exists(path)}")

samsum-train.csv: True
samsum-validation.csv: True
samsum-test.csv: True


In [ ]:
import pandas as pd

train_data = pd.read_csv(train_path)
validation_data = pd.read_csv(validation_path)
test_data = pd.read_csv(test_path)

print("Training dataset shape:", train_data.shape)
print("Validation dataset shape:", validation_data.shape)
print("Testing dataset shape:", test_data.shape)

Training dataset shape: (14732, 3)
Validation dataset shape: (818, 3)
Testing dataset shape: (819, 3)


In [ ]:
print("Columns:", train_data.columns.tolist())
train_data.head()

Columns: ['id', 'dialogue', 'summary']


,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.shape

(14732, 3)

In [ ]:
validation_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=10000, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=700, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(10000, 3)

**Data Pre-Processing**

In [ ]:
import re

def clean_data(text):
    # Handle missing values
    if pd.isna(text):
        return ""

    # Replace newline characters with spaces
    text = re.sub(r"[\r\n]+", " ", text)

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove HTML tags, if any
    text = re.sub(r"<.*?>", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

**Tokenize**

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer for FLAN-T5-base
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
# Raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=data["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    # Handle a single summary
    labels = targets["input_ids"]

    # Ignore padding tokens during loss calculation
    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]

    inputs["labels"] = labels

    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()

validation_dataset = validation_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input ids - dialogue => token ids

# 1 => EOS, 0 => padding

# attention mask
# labels - target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
type(train_dataset)
type(validation_dataset)

list

**Working with our Model**

In [ ]:
from transformers import AutoModelForSeq2SeqLM

# Load the pretrained FLAN-T5-base model
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device)

Device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training configuration
    num_train_epochs=8,
    weight_decay=0.01,
    learning_rate=5e-5,

    # Batch sizes
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Evaluation and checkpoint saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Learning-rate warmup
    warmup_steps=500,

    # Mixed precision
    fp16=False,

    # Logging
    logging_steps=100,    #It tells the model to record training information every 100 training steps.

    # Reproducibility
    seed=42,

    # Keep the best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   #Compare checkpoints using validation loss.
    greater_is_better=False,             #A lower validation loss is better.
)

In [ ]:
from transformers import Trainer   #Trainer class provides an API for feature-complete training in PyTorch
                                   #for most standard use cases.  it is optimised for the transformer models
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset
)

In [ ]:
import torch

model.eval()

batch = train_dataset[:2]                                           #Your test confirms that the freshly loaded FLAN-T5-base model can calculate a valid loss on your training data.

input_ids = torch.tensor(
    [item["input_ids"] for item in batch],
    dtype=torch.long,
    device=device
)

attention_mask = torch.tensor(
    [item["attention_mask"] for item in batch],
    dtype=torch.long,
    device=device
)

labels = torch.tensor(
    [item["labels"] for item in batch],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("Test loss:", outputs.loss.item())
print("Loss is finite:", torch.isfinite(outputs.loss).item())

Test loss: 1.1318296194076538
Loss is finite: True


In [ ]:
# train the model
import os

# Find valid Trainer checkpoints in Google Drive
checkpoints = []

for folder in os.listdir(OUTPUT_DIR):
    checkpoint_path = os.path.join(OUTPUT_DIR, folder)

    if (
        folder.startswith("checkpoint-")
        and os.path.isdir(checkpoint_path)
        and os.path.isfile(
            os.path.join(checkpoint_path, "trainer_state.json")
        )
    ):
        checkpoints.append(checkpoint_path)

# Resume training if a valid checkpoint exists
if checkpoints:
    latest_checkpoint = max(
        checkpoints,
        key=lambda path: int(os.path.basename(path).split("-")[-1])
    )

    print("Resuming training from:", latest_checkpoint)

    trainer.train(resume_from_checkpoint=latest_checkpoint)

else:
    print("No valid checkpoint found.")
    print("Starting training from google/flan-t5-base.")

    trainer.train()

# Report the available checkpoints after training finishes
print("\nAvailable checkpoints in Google Drive:")

for folder in sorted(os.listdir(OUTPUT_DIR)):
    checkpoint_path = os.path.join(OUTPUT_DIR, folder)

    if folder.startswith("checkpoint-") and os.path.isdir(checkpoint_path):
        print(folder)

print("Training cell finished.")

Resuming training from: /content/drive/MyDrive/Dataset_ap/training_checkpoints/checkpoint-17500


[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Epoch,Training Loss,Validation Loss
8,0.965858,1.443187


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



Available checkpoints in Google Drive:
checkpoint-20000
checkpoint-2500
Training cell finished.


In [ ]:
| Epoch | Training loss | Validation loss |
|---:|---:|---:|
| 1 | 1.438920 | 1.386489 |
| 2 | 1.340612 | 1.386756 |
| 3 | 1.259530 | 1.398605 |
| 4 | 1.189744 | 1.398070 |
| 5 | 1.103940 | 1.412740 |
| 6 | 1.075948 | 1.424163 |
| 7 | 1.032966 | 1.440201 |
|8	|0.965858	 |1.443187  |


In [ ]:
import json
import os

checkpoint_path = (
    "/content/drive/MyDrive/Dataset_ap/"
    "training_checkpoints/checkpoint-20000"
)

state_file = os.path.join(checkpoint_path, "trainer_state.json")

with open(state_file, "r") as f:
    trainer_state = json.load(f)

for entry in trainer_state["log_history"]:
    if "loss" in entry or "eval_loss" in entry:
        print(entry)

{'epoch': 0.04, 'grad_norm': 4.287403583526611, 'learning_rate': 9.900000000000002e-06, 'loss': 1.5246510314941406, 'step': 100}
{'epoch': 0.08, 'grad_norm': 5.544394493103027, 'learning_rate': 1.9900000000000003e-05, 'loss': 1.5047015380859374, 'step': 200}
{'epoch': 0.12, 'grad_norm': 4.252424716949463, 'learning_rate': 2.9900000000000002e-05, 'loss': 1.487969970703125, 'step': 300}
{'epoch': 0.16, 'grad_norm': 4.399869441986084, 'learning_rate': 3.99e-05, 'loss': 1.4156292724609374, 'step': 400}
{'epoch': 0.2, 'grad_norm': 5.252821445465088, 'learning_rate': 4.99e-05, 'loss': 1.4566665649414063, 'step': 500}
{'epoch': 0.24, 'grad_norm': 6.618476867675781, 'learning_rate': 4.974615384615385e-05, 'loss': 1.451322021484375, 'step': 600}
{'epoch': 0.28, 'grad_norm': 5.2883076667785645, 'learning_rate': 4.948974358974359e-05, 'loss': 1.464622039794922, 'step': 700}
{'epoch': 0.32, 'grad_norm': 3.5862929821014404, 'learning_rate': 4.9233333333333335e-05, 'loss': 1.528184051513672, 'step':

In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive"))

True


In [ ]:
import os

base_path = "/content/drive/MyDrive/Dataset_ap"

for root, dirs, files in os.walk(base_path):
    if "trainer_state.json" in files:
        print(os.path.join(root, "trainer_state.json"))

/content/drive/MyDrive/Dataset_ap/training_checkpoints/checkpoint-2500/trainer_state.json
/content/drive/MyDrive/Dataset_ap/training_checkpoints/checkpoint-20000/trainer_state.json


In [ ]:
# model load => fine-tune => save the model

In [ ]:
import os

# Define where the trained model will be saved
MODEL_PATH = "/content/drive/MyDrive/Dataset_ap/saved_summary_model"

# Create the folder if it doesn't exist
os.makedirs(MODEL_PATH, exist_ok=True)

# Save the trained model and tokenizer
trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("Final model saved to:", MODEL_PATH)
print("Saved files:", os.listdir(MODEL_PATH))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/Dataset_ap/saved_summary_model
Saved files: ['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()

print("Model loaded successfully on:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully on: cuda


**Test The Core Logic For Summarization**

In [ ]:
def summarize_dialogue(dialogue):
    # Step 1: Clean the input dialogue
    dialogue = clean_data(dialogue)

    # Step 2: Tokenize the dialogue
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Step 3: Generate the summary
    model.eval()  # Switch to evaluation mode
    model.to(device)

    with torch.no_grad():
        targets = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=150,
            num_beams=4,
            early_stopping=True
        )

    # Step 4: Convert generated token IDs into readable text
    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  Artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.
